# Data Loading

In [2]:
from pathlib import Path
import zipfile
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

## Dataset 1 *Occupational Employment and Wage Statistics Metros* 

In [3]:
zip_path = RAW / "oews_metro_may2025.zip"
oews_dir = INTERIM / "oews_metro_may2025"

if not oews_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(oews_dir)

for p in sorted(oews_dir.rglob("*")):
    if p.is_file():
        print(p.relative_to(oews_dir))

oesm25ma\BOS_M2025_dl.xlsx
oesm25ma\file_descriptions.xlsx
oesm25ma\MSA_M2025_dl.xlsx


In [4]:
oews_path = max(
    (p for p in oews_dir.rglob("*.xlsx")
     if "descriptions" not in p.name.lower()),
    key=lambda p: p.stat().st_size,
)
print("Loading:", oews_path.name)

oews = pd.read_excel(
    oews_path,
    dtype={"AREA": str, "OCC_CODE": str},
    na_values=["*", "**"],   # OEWS suppression markers
)

Loading: MSA_M2025_dl.xlsx


## Dataset 2 *Fair Market Rents*

In [5]:
fmr = pd.read_csv(RAW / "hud_fmr_fy2026.xlsx")

## Data Assessment

In [6]:
print(oews.shape, "|", fmr.shape)
display(oews.head(3))
display(fmr.head(3))

# Data scientists only (SOC 15-2051)
ds = oews[oews["OCC_CODE"] == "15-2051"].copy()
print(f"{len(ds)} metro/area rows for data scientists")
print(ds[["A_MEDIAN"]].describe())
print("Missing median wage:", ds["A_MEDIAN"].isna().sum())

(150023, 32) | (2622, 10)


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,21.32,29.92,45.19,24900,30980,44340,62230,93990,NaN,NaN
1,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,45.9,64.34,92.98,47790,64990,95480,133840,193390,NaN,NaN
2,10180,"Abilene, TX",4,TX,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,79.7,113.51,187.68,88400,121980,165770,236100,390370,NaN,NaN


,OBJECTID,FMR_CODE,FMR_AREANAME,FMR_0BDR,FMR_1BDR,FMR_2BDR,FMR_3BDR,FMR_4BDR,Shape__Area,Shape__Length
0,1,METRO10180M10180,"Abilene, TX MSA",1010,1020,1276,1653,1987,1.005900e+10,461477.288078
1,2,METRO10380M10380,"Aguadilla, PR MSA",480,482,564,680,756,8.612798e+08,164337.142741
2,3,METRO10420M10420,"Akron, OH MSA",904,985,1268,1547,1681,4.225362e+09,291486.075708


291 metro/area rows for data scientists
        A_MEDIAN
count        282
unique       273
top        92800
freq           2
Missing median wage: 9


There are 273 usable rows in the OES data set for the Data Scientist occupation's median wage. These rows represent metro areas across the US. 
The FMR dataframe has 2622 rows, which represent all the areas in the US that have FMR data

In [14]:
display(ds['AREA'].unique())
display(fmr['FMR_CODE'].unique())

<StringArray>
['10180', '10420', '10500', '10580', '10740', '10900', '11020', '11100',
 '11180', '11200',
 ...
 '48140', '48620', '48700', '48900', '49020', '49180', '49340', '49620',
 '49660', '49740']
Length: 291, dtype: str

<StringArray>
['METRO10180M10180', 'METRO10380M10380', 'METRO10420M10420',
 'METRO10500M10500', 'METRO10540M10540', 'METRO10580M10580',
 'METRO10740M10740', 'METRO10780M10780', 'METRO10900M10900',
 'METRO10900MM5640',
 ...
 'NCNTY66010N66010', 'NCNTY69999N69999', 'NCNTY72001N99999',
 'NCNTY72081N99999', 'NCNTY72083N99999', 'NCNTY72141N99999',
 'NCNTY72923N72923', 'NCNTY78010N78010', 'NCNTY78020N78020',
 'NCNTY78030N78030']
Length: 2622, dtype: str

In [ ]:
metros = pd.DataFrame()
for code in fmr['FMR_CODE']:
    for area in ds['AREA']:
        if area in code:
            metros = pd.concat([metros, fmr[fmr['FMR_CODE'] == code].copy()], ignore_index=True)

display(metros.shape)
display(metros['FMR_CODE'].duplicated().sum())

(527, 10)

np.int64(0)

In [42]:
metros.sample(10, random_state=10)

,OBJECTID,FMR_CODE,FMR_AREANAME,FMR_0BDR,FMR_1BDR,FMR_2BDR,FMR_3BDR,FMR_4BDR,Shape__Area,Shape__Length
410,519,METRO41100M41100,"St. George, UT MSA",1210,1218,1575,2072,2624,9.961387e+09,436673.632942
358,449,METRO36540M36540,"Omaha-Council Bluffs, NE-IA HUD Metro FMR Area",1090,1148,1368,1813,2046,1.675469e+10,680720.253883
399,504,METRO40060M51097,"King and Queen County, VA HUD Metro FMR Area",1089,1139,1250,1536,1929,1.330367e+09,302258.282498
24,30,METRO12060N13211,"Morgan County, GA HUD Metro FMR Area",1256,1315,1442,1729,2419,1.327085e+09,165278.411706
302,375,METRO31540N55049,"Iowa County, WI HUD Metro FMR Area",884,890,1168,1535,1959,3.720941e+09,261542.144078
173,207,METRO23540M23540,"Gainesville, FL HUD Metro FMR Area",1154,1246,1493,1868,1977,4.559561e+09,420670.009825
78,94,METRO16300M16300,"Cedar Rapids, IA HUD Metro FMR Area",738,816,1071,1490,1797,3.409380e+09,234905.632611
47,58,METRO13820M13820,"Birmingham-Hoover, AL HUD Metro FMR Area",1024,1155,1266,1583,1801,1.443118e+10,832270.238893
184,226,METRO24580N55083,"Oconto County, WI HUD Metro FMR Area",723,793,973,1272,1288,5.268761e+09,437440.545978
151,176,METRO20740M20740,"Eau Claire, WI MSA",932,962,1181,1582,1588,8.717659e+09,384539.112367


FMR_CODE is more granular than the AREA code in the OES data. FMR has the area, then a subdivision of that area. This demands an analytical decision that will have a meaningful effect on the inferences. I could take the average market rents across all subdivisions of the metro, use the lowest, or some measure of center.